Run both benchmarks before running this plotting notebook. From the repository root on a CUDA GPU:

```bash
pixi run python -m analysis.inference_efficiency --output logs/inference
pixi run python -m analysis.inference_efficiency --planning --config conf/analysis/planning_efficiency.json --output logs/planning
```

Replace `inference_run` and `planning_run` below with your output directories. Each must contain all five methods. Alternatively, set `timings_file` to a previously exported `timings.json`. Run plotting on CPU from the repository root or `analysis/`. Nimbus Roman is used when installed; otherwise the default serif font is used. Exports use a fresh `logs/paper_figures/` directory; new benchmark values may differ from the paper's historical timings.


In [ ]:
%matplotlib inline
import os
import sys
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.transforms import offset_copy
from matplotlib.patches import Rectangle

root = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from analysis.artifacts import create_output

for font in Path('/usr/share/fonts/urw-base35').glob('NimbusRoman-*.otf'):
    font_manager.fontManager.addfont(str(font))
try:
    font_manager.findfont('Nimbus Roman', fallback_to_default=False)
    font_family = 'Nimbus Roman'
except ValueError:
    font_family = 'serif'
    print('Nimbus Roman not found; using the default serif font, so text layout may differ from the paper.')
plt.rcdefaults()
plt.rcParams.update({'font.family': font_family, 'mathtext.fontset': 'stix',
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.linewidth': 1})

# Read your paired benchmark runs, or set timings_file to a saved timings.json.
timings_file = None
inference_run = Path('PATH_TO_YOUR_INFERENCE_RUN')
planning_run = Path('PATH_TO_YOUR_PLANNING_RUN')


## Read saved results

Read only the files selected above; no model inference or simulation.


In [ ]:
from analysis.timing_results import read_run_timings

methods = ['vanilla', 'long_history', 'prefix_context_joint', 'privileged_physics', 'context']
if timings_file is not None:
    rows = json.loads(timings_file.read_text())
    inputs = {'timings': str(timings_file.resolve())}
else:
    rows = read_run_timings(inference_run, planning_run, methods)
    inputs = {kind: {method: str((directory / f'{method}.json').resolve()) for method in methods}
              for kind, directory in [('inference', inference_run), ('planning', planning_run)]}
if len(rows) != len(methods) or {row['method'] for row in rows} != set(methods):
    raise ValueError('Timing results must contain exactly the five plotted methods')
rows = sorted(rows, key=lambda row: methods.index(row['method']))
for row in rows:
    if not np.isfinite([row['rollout_ms'], row['planning_s'], row['rollout_mean_ms'], row['planning_mean_s']]).all():
        raise ValueError(f"Non-finite timings for {row['method']}")
    print(row)

run_dir = create_output(root / 'logs/paper_figures' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'), {'inputs': inputs})


## Plot and export

Keep the paper layout and export PDF/PNG and plotted values into a fresh output directory.


In [ ]:
page_width, page_height = 899.271331, 280.576875
fig = plt.figure(figsize=(page_width / 72, page_height / 72))
colors = ['#aad2c8', '#f78c6b', '#06d6a0', '#42a5f5', '#ff647b']
labels = ['DINO-WM', '+ Long History', '+ Context Prefix',
          r'+ Privileged Physics$^{\ddag}$', '+ PhysiCal (Ours)']
y = np.arange(5) * .97
for left, metric, maximum, ticks, padding, title, xlabel in [
    (13.2, 'rollout_ms', 150, [0, 50, 100, 150], 1.5, '10-Step Open-Loop Rollout', 'Inference Time (ms)'),
    (473.8330967967, 'planning_s', 520, [0, 200, 400], 5, 'Planning', 'Planning Time (s)'),
]:
    ax = fig.add_axes([left / page_width, 63.590625 / page_height,
                       408.9475883263 / page_width, 183.88 / page_height])
    values = [row[metric] for row in rows]
    ax.barh(y, values, color=colors, height=.8)
    ax.set(xlim=(0, maximum), ylim=(4.44, -.56), yticks=[])
    ax.set_xticks(ticks)
    ax.tick_params(axis='x', labelsize=24, pad=3.71875)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', linestyle=':', alpha=.5, linewidth=1)
    ax.set_title(title, fontsize=26, pad=8)
    ax.set_xlabel(xlabel, fontsize=26, labelpad=4.125 if metric == 'rollout_ms' else 4.140625)
    for i, (value, label) in enumerate(zip(values, labels)):
        # Match the original PDF text baseline across Matplotlib versions.
        baseline = 0 if i == 3 else (-.0859375 if i == 4 else -.109375)
        transform = offset_copy(ax.transData, fig=fig, y=baseline, units='points')
        ax.text(value + padding, y[i], label, va='center', fontsize=24, transform=transform,
                color=colors[i] if i == 4 else '#222222',
                fontweight='bold' if i == 4 else 'normal')
for extension in ('pdf', 'png'):
    fig.savefig(run_dir / f'7-inference-time-v2.{extension}', dpi=200)
(run_dir / 'timings.json').write_text(json.dumps(rows, indent=2) + '\n')
plt.show()
print(run_dir)


Paper styling is retained; plotted values come from the selected inputs. PDF file bytes may differ because of metadata.
